In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.features import build_features, COLUMN_ORDER
from src.models import RFCModel, XGBModel, XGBModel2

In [4]:

df_raw = load_data('SPY')
X_raw = build_features(df_raw)

df_spy = X_raw.copy()

ma50  = df_raw['Close'].rolling(50).mean()
ma200 = df_raw['Close'].rolling(200).mean()
gc    = (ma50 > ma200).astype(int)
gc_clean = gc.reset_index(drop=True)

transition = np.zeros(len(df_spy), dtype=int)
label = gc_clean.iloc[0]
for i in range(1, len(gc_clean)):
    if gc_clean.iloc[i] != label:
        label = gc_clean.iloc[i]
        start = max(0, i - 30)
        transition[start:i] = 1

df_spy['Transition'] = transition

df_spy = df_spy.dropna().reset_index(drop=True)


X = df_spy[COLUMN_ORDER]
y = df_spy['Transition']

split = int(len(df_spy) * 0.75)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

print(f"Train size: {len(X_train)}  |  positives: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test size:  {len(X_test)}   |  positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
X_train.head(500)

Train size: 4563  |  positives: 528 (11.6%)
Test size:  1521   |  positives: 150 (9.9%)


,Return,Volatility,Cumulated_Return_5d,RSI14,Volume_ROC,ATR,VIX_spike,Distance_GC,MA_velocity,MA50_slope,Distance_normalized,MA_cross_momentum
0,0.004467,0.016713,0.016960,46.488280,-43.405512,2.234626,1.204567,0.008671,-0.559687,-0.011881,0.897102,56.402279
1,-0.006671,0.016545,0.035946,42.632307,-6.475955,2.254458,1.181164,0.007573,-0.525658,-0.012497,0.773324,78.072778
2,-0.023506,0.017220,0.015363,37.116720,100.083207,2.342284,1.290078,0.006090,-0.496997,-0.013276,0.612879,111.846077
3,0.002751,0.016896,-0.018843,42.134899,-6.681045,2.284914,1.375873,0.004669,-0.527376,-0.013361,0.463346,136.775979
4,0.018976,0.017493,-0.004468,48.775247,116.568323,2.317494,1.265942,0.003368,-0.554406,-0.013777,0.329125,185.008178
...,...,...,...,...,...,...,...,...,...,...,...,...
495,-0.024238,0.027371,0.108195,51.334742,17.412655,2.021278,1.000913,-0.133283,0.337903,-0.009746,-6.334750,3.166481
496,0.019873,0.026973,0.094754,59.368733,6.010676,2.010570,0.950624,-0.132059,0.454877,-0.009155,-6.282247,3.994448
497,0.004192,0.026969,0.053232,61.863530,-35.087501,2.023605,0.937234,-0.131280,0.475285,-0.008949,-6.256594,4.157024
498,0.017261,0.027022,0.065462,58.405791,-31.755362,1.959829,0.926530,-0.130295,0.476080,-0.007727,-6.220196,3.756739


In [7]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

f1_transition = make_scorer(f1_score, zero_division=0)

param_dist = {
    'n_estimators':     randint(200, 600),
    'max_depth':        randint(3, 8),
    'learning_rate':    uniform(0.01, 0.09),
    'subsample':        uniform(0.5, 0.4),
    'colsample_bytree': uniform(0.5, 0.4),
    'min_child_weight': randint(1, 10),      
    'gamma':            uniform(0, 0.3),     
}

tscv = TimeSeriesSplit(n_splits=5)
scale_pos_weight = 4.26

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=30,   
    verbosity=0,
    random_state=42
)


last_train_idx, last_val_idx = list(tscv.split(X_train))[-1]
eval_set = [(X_train.iloc[last_val_idx], y_train.iloc[last_val_idx])]

rs = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring=f1_transition,
    cv=tscv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=False          
)

rs.fit(X_train, y_train,
       eval_set=eval_set,
       verbose=False)

print("Best params :", rs.best_params_)
print("Best CV F1  :", round(rs.best_score_, 3))

best = rs.best_params_
xgb_tuned = XGBClassifier(
    **best,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=30,
    verbosity=0,
    random_state=42
)
xgb_tuned.fit(
    X_train.iloc[last_train_idx], y_train.iloc[last_train_idx],
    eval_set=eval_set,
    verbose=False
)

print(f"Best n_estimators (early stopping) : {xgb_tuned.best_iteration}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params : {'colsample_bytree': np.float64(0.5020739451095947), 'gamma': np.float64(0.18836832448459082), 'learning_rate': np.float64(0.0274846558160838), 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 342, 'subsample': np.float64(0.6502331810559776)}
Best CV F1  : 0.641
Best n_estimators (early stopping) : 179


In [8]:

xgb = XGBModel2(
            n_estimators=179,
            max_depth=6,
            learning_rate=0.0275,
            subsample=0.650,
            colsample_bytree=0.502,
            gamma=0.188,
            min_child_weight=3,
            eval_metric='logloss',
            verbosity=0,
            scale_pos_weight = 4.26 #simulates 19% pos to compare w EEm
        )

xgb.fit(X_train, y_train)

y_proba_xgb = xgb.predict_proba(X_test)
print("Threshold | Precision | Recall |  F1   | N_pred")
print("-" * 52)
for t in np.arange(0.3, 0.95, 0.05):
    pred = (y_proba_xgb >= t).astype(int)
    if pred.sum() > 0:
        p = precision_score(y_test, pred, zero_division=0)
        r = recall_score(y_test, pred, zero_division=0)
        f = f1_score(y_test, pred, zero_division=0)
        print(f"  {t:.2f}    |   {p:.3f}   |  {r:.3f} | {f:.3f} | {pred.sum()}")

Threshold | Precision | Recall |  F1   | N_pred
----------------------------------------------------
  0.30    |   0.460   |  0.887 | 0.606 | 289
  0.35    |   0.494   |  0.867 | 0.630 | 263
  0.40    |   0.510   |  0.827 | 0.631 | 243
  0.45    |   0.528   |  0.807 | 0.639 | 229
  0.50    |   0.556   |  0.800 | 0.656 | 216
  0.55    |   0.609   |  0.800 | 0.692 | 197
  0.60    |   0.648   |  0.787 | 0.711 | 182
  0.65    |   0.685   |  0.767 | 0.723 | 168
  0.70    |   0.711   |  0.753 | 0.731 | 159
  0.75    |   0.754   |  0.713 | 0.733 | 142
  0.80    |   0.808   |  0.647 | 0.719 | 120
  0.85    |   0.905   |  0.573 | 0.702 | 95
  0.90    |   0.931   |  0.447 | 0.604 | 72


In [9]:
rfc = RFCModel(n_estimators=346,
            max_depth=9,
            min_samples_split=17,
            min_samples_leaf=5,
            max_features=0.5463,
        )
rfc.fit(X_train, y_train)

y_proba_rfc = rfc.predict_proba(X_test)
print("Threshold | Precision | Recall |  F1   | N_pred")
print("-" * 52)
for t in np.arange(0.3, 0.95, 0.05):
    pred = (y_proba_rfc >= t).astype(int)
    if pred.sum() > 0:
        p = precision_score(y_test, pred, zero_division=0)
        r = recall_score(y_test, pred, zero_division=0)
        f = f1_score(y_test, pred, zero_division=0)
        print(f"  {t:.2f}    |   {p:.3f}   |  {r:.3f} | {f:.3f} | {pred.sum()}")

Threshold | Precision | Recall |  F1   | N_pred
----------------------------------------------------
  0.30    |   0.466   |  0.900 | 0.614 | 290
  0.35    |   0.475   |  0.893 | 0.620 | 282
  0.40    |   0.498   |  0.880 | 0.636 | 265
  0.45    |   0.508   |  0.880 | 0.644 | 260
  0.50    |   0.531   |  0.847 | 0.653 | 239
  0.55    |   0.557   |  0.820 | 0.663 | 221
  0.60    |   0.624   |  0.787 | 0.696 | 189
  0.65    |   0.735   |  0.720 | 0.727 | 147
  0.70    |   0.786   |  0.660 | 0.717 | 126
  0.75    |   0.830   |  0.620 | 0.710 | 112
  0.80    |   0.848   |  0.447 | 0.585 | 79
  0.85    |   0.811   |  0.287 | 0.424 | 53
  0.90    |   0.692   |  0.060 | 0.110 | 13


In [10]:
def compute_transition(df_raw):
    ma50  = df_raw['Close'].rolling(50).mean()
    ma200 = df_raw['Close'].rolling(200).mean()
    gc    = (ma50 > ma200).astype(int)
    transition = np.zeros(len(df_raw), dtype=int)
    label = gc.iloc[0]
    for i in range(1, len(gc)):
        if gc.iloc[i] != label:
            label = gc.iloc[i]
            start = max(0, i - 30)
            transition[start:i] = 1
    return pd.Series(transition, name='Transition')


def prepare_asset(ticker, split=0.75):
    df_raw = load_data(ticker)
    X_raw  = build_features(df_raw)
    df     = X_raw.copy()
    df['Transition'] = compute_transition(df_raw).values
    df     = df.dropna().reset_index(drop=True)
    split_idx = int(len(df) * split)
    X = df[COLUMN_ORDER]
    y = df['Transition']
    return X.iloc[split_idx:], y.iloc[split_idx:]


def validate_cross_assets(rfc_model, xgb_model, thresh_rfc, thresh_xgb, tickers):
    results = []
    for ticker in tickers:
        X_test, y_test = prepare_asset(ticker)
        print(f"\n{'='*50}")
        print(f"Asset: {ticker} | positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

        for model, thresh, name in [
            (rfc_model, thresh_rfc, 'RFC'),
            (xgb_model, thresh_xgb, 'XGB')
        ]:
            proba = model.predict_proba(X_test)
            pred  = (proba >= thresh).astype(int)
            results.append({
                'Asset':     ticker,
                'Model':     name,
                'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
                'Recall':    round(recall_score(y_test, pred, zero_division=0), 3),
                'F1':        round(f1_score(y_test, pred, zero_division=0), 3),
                'N_pred':    int(pred.sum())
            })

    df_results = pd.DataFrame(results)
    print("\n" + "="*60)
    print("CROSS-ASSET VALIDATION SUMMARY")
    print("="*60)
    print(df_results.to_string(index=False))
    print(f"\nMean F1 RFC: {df_results[df_results['Model']=='RFC']['F1'].mean():.3f}")
    print(f"Mean F1 XGB: {df_results[df_results['Model']=='XGB']['F1'].mean():.3f}")
    return df_results


results = validate_cross_assets(rfc, xgb, thresh_rfc=0.65, thresh_xgb=0.75, tickers=['SPY', 'DIA', 'QQQ'])


Asset: SPY | positives: 150 (9.9%)

Asset: DIA | positives: 157 (10.3%)

Asset: QQQ | positives: 132 (8.7%)

CROSS-ASSET VALIDATION SUMMARY
Asset Model  Precision  Recall    F1  N_pred
  SPY   RFC      0.735   0.720 0.727     147
  SPY   XGB      0.754   0.713 0.733     142
  DIA   RFC      0.609   0.713 0.657     184
  DIA   XGB      0.697   0.675 0.686     152
  QQQ   RFC      0.805   0.780 0.792     128
  QQQ   XGB      0.911   0.697 0.790     101

Mean F1 RFC: 0.725
Mean F1 XGB: 0.736


In [12]:
results = validate_cross_assets(rfc, xgb, thresh_rfc=0.65, thresh_xgb=0.75, tickers=['GLD', 'TLT', 'USO', 'VNQ', 'URTH', 'EEM', 'EWJ', 'CAC'])


Asset: GLD | positives: 273 (22.5%)

Asset: TLT | positives: 264 (19.4%)

Asset: USO | positives: 180 (16.0%)

Asset: VNQ | positives: 243 (19.9%)

Asset: URTH | positives: 60 (7.9%)

Asset: EEM | positives: 206 (15.7%)

Asset: EWJ | positives: 184 (12.1%)

Asset: CAC | positives: 210 (13.8%)

CROSS-ASSET VALIDATION SUMMARY
Asset Model  Precision  Recall    F1  N_pred
  GLD   RFC      0.722   0.714 0.718     270
  GLD   XGB      0.824   0.601 0.695     199
  TLT   RFC      0.589   0.712 0.645     319
  TLT   XGB      0.778   0.598 0.677     203
  USO   RFC      0.527   0.539 0.533     184
  USO   XGB      0.798   0.394 0.528      89
  VNQ   RFC      0.667   0.716 0.690     261
  VNQ   XGB      0.786   0.605 0.684     187
 URTH   RFC      0.594   0.683 0.636      69
 URTH   XGB      0.688   0.550 0.611      48
  EEM   RFC      0.695   0.354 0.469     105
  EEM   XGB      0.969   0.301 0.459      64
  EWJ   RFC      0.409   0.538 0.465     242
  EWJ   XGB      0.465   0.359 0.405     14